# Dataset Exploration: databricks-dolly-15k vs alpaca-cleaned
Цель: найти маркетинговые примеры и выбрать лучший датасет.

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset
import pandas as pd

MARKETING_KEYWORDS = [
    'marketing', 'advertisement', 'ad copy', 'slogan', 'tagline',
    'campaign', 'brand', 'copywriting', 'promotional', 'email subject',
    'call to action', 'cta', 'product description', 'landing page',
    'social media post', 'headline', 'pitch', 'write an ad', 'write a post'
]

def is_marketing(text):
    text_lower = str(text).lower()
    return any(kw in text_lower for kw in MARKETING_KEYWORDS)

print('Keywords loaded:', len(MARKETING_KEYWORDS))

## 1. databricks/databricks-dolly-15k

In [ ]:
print('Loading dolly-15k...')
dolly = load_dataset('databricks/databricks-dolly-15k', split='train')
print(f'Total: {len(dolly)}')
print(f'Columns: {dolly.column_names}')

In [ ]:
dolly_df = dolly.to_pandas()
print('Categories:', dolly_df['category'].value_counts().to_dict())

dolly_marketing = dolly_df[
    dolly_df['instruction'].apply(is_marketing) |
    dolly_df['context'].apply(is_marketing)
]
print(f'\nMarketing examples: {len(dolly_marketing)} / {len(dolly_df)}')

In [ ]:
print('=== ПРИМЕРЫ dolly-15k ===')
for i, row in dolly_marketing.head(3).iterrows():
    print(f'\n[{i}] CATEGORY: {row["category"]}')
    print(f'    INSTRUCTION: {row["instruction"][:300]}')
    print(f'    RESPONSE: {row["response"][:300]}')

## 2. yahma/alpaca-cleaned

In [ ]:
print('Loading alpaca-cleaned...')
alpaca = load_dataset('yahma/alpaca-cleaned', split='train')
print(f'Total: {len(alpaca)}')
print(f'Columns: {alpaca.column_names}')

In [ ]:
alp_df = alpaca.to_pandas()
alp_marketing = alp_df[
    alp_df['instruction'].apply(is_marketing) |
    alp_df['input'].apply(is_marketing)
]
print(f'Marketing examples: {len(alp_marketing)} / {len(alp_df)}')

In [ ]:
print('=== ПРИМЕРЫ alpaca-cleaned ===')
for i, row in alp_marketing.head(3).iterrows():
    print(f'\n[{i}] INSTRUCTION: {row["instruction"][:300]}')
    if row['input']:
        print(f'    INPUT: {str(row["input"])[:150]}')
    print(f'    OUTPUT: {row["output"][:300]}')

## 3. Итог

In [ ]:
print('=' * 50)
print('ИТОГ')
print('=' * 50)
print(f'dolly-15k      — маркетинг: {len(dolly_marketing)}')
print(f'alpaca-cleaned — маркетинг: {len(alp_marketing)}')
print()
for name, count in [('dolly-15k', len(dolly_marketing)), ('alpaca-cleaned', len(alp_marketing))]:
    status = 'достаточно' if count >= 200 else f'мало — нужно добрать {200 - count}'
    print(f'{name}: {count} примеров — {status}')

avg_dolly = dolly_marketing['response'].apply(lambda x: len(str(x).split())).mean()
avg_alp = alp_marketing['output'].apply(lambda x: len(str(x).split())).mean()
print(f'\ndolly-15k      средняя длина ответа: {avg_dolly:.0f} слов')
print(f'alpaca-cleaned средняя длина ответа: {avg_alp:.0f} слов')